# Is the Judge the Bottleneck? Run Condition D on Kaggle

This notebook runs **Condition D** (`pipeline/Methods/condition_d.py`) on Kaggle's GPU, with exactly the code, prompts
and settings of the GitHub branch `SC.PV2`. In Condition D the **Oracle judge** decides when to stop (once every Gold
paragraph is in the Evidence) and says what is missing (the first missing Gold title), and that title is the next Round's search query.
Pipeline v1: no Rewriter, so each Round after the first searches with the Steer signal as written,
and k = 3 new paragraphs per Round (from `config.yaml`), so 3 Rounds give at most 9.
The LLM judge is still asked every Round, and its verdict is logged next to
the Oracle's, but it changes nothing. Condition D is the best case of the 2x2. The model runs through **Ollama**, the
same version and the same model file as on the Mac, so Kaggle's answers match the Mac's.

**Before you run it**
1. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. In the right-hand panel (**Settings**): **Accelerator: GPU T4 x2** and **Internet: On** (Internet needs a
   phone-verified Kaggle account).
3. **Add Input → Datasets → `shoimyachy/hotpotqa-bm25-index`** (our search index; Condition D searches every Round).
4. Cell 2 is set to all 1,000 Test questions; change it there if needed.
5. **Save Version → Save & Run All (Commit)**. The notebook then runs in the background, even with the browser closed,
   for up to 12 hours.

Long lists of yellow "dependency conflict" warnings during installs are normal on Kaggle; only a red error that stops a
cell matters. Every check stops the notebook with a clear message if something is wrong.

**On the Mac** (to try it before Kaggle): open this notebook in VS Code and choose `.venv` as the kernel (top right),
so it uses the project's own libraries.

**What you get** (right-hand panel, **Output**, once it has finished)
- `condition_d_logs_<date>.zip`: this run's log, `<HHMMSS>.txt` (header, one block per question, summary). Each Round
  shows the search, the Oracle's decision and steer, and the LLM judge's verdict with "agrees" or "DISAGREES".
- The same log unzipped under `Is-the-Judge-the-Bottleneck/runs/condition_d/<date>/`.

**How long it takes:** at most 4 model calls per question (an LLM judge call per Round, one Answerer call), fewer
when the Oracle stops early.

## 1. Settings

Change only the lines in this cell. Everything else (prompts, model, k, Rounds, temperature, seed) comes from the
project's own files on GitHub.

In [ ]:
# Here we choose the questions: "pilot" (100) or "test" (1,000), and how many from the start (None = all of them).
QUESTION_SET = "test"
HOW_MANY_QUESTIONS = None

# Here we choose whether the log also holds every call's full prompt and reply (makes the log much longer).
LOG_FULL_PROMPTS = False

# Here is where the code comes from.
REPOSITORY_ADDRESS = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
BRANCH = "SC.PV2"

# Here we name the Ollama version the Mac uses, and the exact model file the Mac uses. A model's ID is the first
# 12 characters of its digest, as `ollama list` shows it.
OLLAMA_VERSION_ON_THE_MAC = "0.35.1"
EXPECTED_MODEL_IDS = {"qwen3:8b": "500a1f067a9f"}

## 2. Check the machine

Without a GPU the model would run on the CPU and take days; without internet nothing can be downloaded. Either way, the
notebook stops here with a clear message.

In [ ]:
import json
import os
import shutil
import socket
import subprocess
import sys
import urllib.request
from datetime import datetime
from pathlib import Path

# Here we tell the machines apart the same way pipeline/settings.py does: Kaggle sets KAGGLE_KERNEL_RUN_TYPE.
RUNNING_ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("Machine:", "Kaggle" if RUNNING_ON_KAGGLE else "this computer")


def check_the_gpu_is_switched_on() -> None:
    """On Kaggle, stop with a clear message if no GPU is switched on; on the Mac, there is nothing to check."""
    if not RUNNING_ON_KAGGLE:
        print("On the Mac: Ollama uses the Mac's own GPU, nothing to check here.")
        return
    # Here we look for nvidia-smi, the program that comes with the GPU driver; without it there is no GPU.
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No GPU found. In the notebook's Settings, set Accelerator to 'GPU T4 x2', then run again.")
    # Here we print each GPU's name and memory, so the output shows what the run used.
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], check=True)


def check_the_internet_is_on() -> None:
    """Stop with a clear message if this machine can't reach the internet."""
    # Here we look up github.com; if that fails, nothing can be downloaded.
    try:
        socket.gethostbyname("github.com")
        print("Internet: on")
    except OSError:
        raise RuntimeError("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                           "then switch Internet on in the notebook's Settings and run again.")


check_the_gpu_is_switched_on()
check_the_internet_is_on()

## 3. Get the project code

On Kaggle this downloads branch `BRANCH` from GitHub (or, if it is already here, resets it to exactly what GitHub has),
then installs the project's libraries. On the Mac it uses the repo this notebook sits in.

In [ ]:
KAGGLE_PROJECT_FOLDER = Path("/kaggle/working/Is-the-Judge-the-Bottleneck")


def find_or_clone_project_folder() -> Path:
    """On Kaggle, clone the project (or reset it to GitHub's version); on the Mac, find the repo above this notebook."""
    if not RUNNING_ON_KAGGLE:
        # Here we go up from the notebook's folder until we reach the folder that holds config.yaml.
        for folder in [Path.cwd(), *Path.cwd().parents]:
            if (folder / "config.yaml").exists():
                return folder
        raise FileNotFoundError("No project folder (a folder with config.yaml) was found above this notebook.")

    if KAGGLE_PROJECT_FOLDER.exists():
        # Here we reset an existing copy to exactly what GitHub has on BRANCH, throwing away any stray edits.
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "checkout", "--force", "-B", BRANCH,
                        f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPOSITORY_ADDRESS, str(KAGGLE_PROJECT_FOLDER)], check=True)
    return KAGGLE_PROJECT_FOLDER


PROJECT_FOLDER = find_or_clone_project_folder()
os.chdir(PROJECT_FOLDER)

# Here, on Kaggle, we install the exact library versions the project needs (the Mac's .venv already has them).
if RUNNING_ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Here we let Python import the project's code (pipeline/) from the project folder straight away.
sys.path.insert(0, str(PROJECT_FOLDER))

# Here we print which commit of the code this run uses, so every result can be traced back to its code.
last_commit = subprocess.run(["git", "log", "-1", "--format=%h %s (%ci)"], capture_output=True, text=True)
print(f"Project folder: {PROJECT_FOLDER}")
print(f"Code version:   {last_commit.stdout.strip()}")

## 4. Install Ollama (Kaggle only)

Kaggle starts every session from a clean machine, so Ollama is installed each time, in exactly the Mac's version. Two
small tools are installed first if missing: `zstd` (the Ollama installer unpacks with it) and `lsof` (the project's
stop function finds the server with it).

In [ ]:
def install_missing_system_tools() -> None:
    """Install zstd and lsof with apt-get, but only the ones that are missing."""
    missing_tools = [tool for tool in ["zstd", "lsof"] if shutil.which(tool) is None]
    if not missing_tools:
        print("zstd and lsof are already installed.")
        return
    print(f"Installing {', '.join(missing_tools)} ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *missing_tools], check=True)


def installed_ollama_version() -> str | None:
    """The installed Ollama version, for example "0.35.1", or None if Ollama is not installed."""
    if shutil.which("ollama") is None:
        return None
    # Here we ask Ollama for its version; its last word is always this program's own version.
    version_output = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    return (version_output.stdout + version_output.stderr).strip().split()[-1]


def install_ollama(version: str) -> None:
    """Install the given Ollama version with Ollama's own install script, unless that version is already installed."""
    if installed_ollama_version() == version:
        print(f"Ollama {version} is already installed.")
        return
    # Here we pass the version to the install script, so Kaggle gets exactly the Mac's version.
    install_environment = dict(os.environ, OLLAMA_VERSION=version)
    # Here "pipefail" makes the step fail if downloading the install script fails, instead of silently doing nothing.
    install_command = "set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh"
    subprocess.run(["bash", "-c", install_command], env=install_environment, check=True)


if RUNNING_ON_KAGGLE:
    install_missing_system_tools()
    install_ollama(OLLAMA_VERSION_ON_THE_MAC)

# Here we stop on Kaggle if the version is not the Mac's, because a different version can answer differently.
ollama_version_in_use = installed_ollama_version()
print(f"Ollama version in use: {ollama_version_in_use} (Mac: {OLLAMA_VERSION_ON_THE_MAC})")
if RUNNING_ON_KAGGLE and ollama_version_in_use != OLLAMA_VERSION_ON_THE_MAC:
    raise RuntimeError(f"Ollama {ollama_version_in_use} is installed, but the Mac uses {OLLAMA_VERSION_ON_THE_MAC}.")

## 5. Start the server, download the model and check it

The server runs on this project's own port (see `pipeline/settings.py`). On Kaggle the model goes to `/tmp/ollama`
(not saved with the notebook) and downloads in a few minutes. The notebook then checks it is the Mac's exact model file,
and, on Kaggle, that it sits fully on the GPU.

In [ ]:
from pipeline import llm2, load_config, settings

CONFIG = load_config()
MODEL_NAMES = sorted(set(CONFIG["models"].values()))


def model_ids_on_this_server() -> dict:
    """Each model's ID on this project's server, for example {"qwen3:8b": "500a1f067a9f"}."""
    with urllib.request.urlopen(settings.OLLAMA_ADDRESS + "/api/tags", timeout=10) as response:
        models = json.load(response)["models"]
    # Here we keep the first 12 characters of each digest, the ID `ollama list` shows.
    return {model["name"]: model["digest"][:12] for model in models}


def check_the_models_are_the_macs() -> None:
    """Stop if a model is not the exact file the Mac uses, because a different file can answer differently."""
    ids_here = model_ids_on_this_server()
    for model_name, expected_id in EXPECTED_MODEL_IDS.items():
        id_here = ids_here.get(model_name)
        if id_here != expected_id:
            raise RuntimeError(f"{model_name} is not the Mac's model file (Mac: {expected_id}, here: {id_here}).")
        print(f"{model_name}: the Mac's model file ({id_here}).")


def gpu_share_of_model(model_name: str) -> int:
    """Load the model with a small call, then return how much of it sits on the GPU, in percent."""
    # Here we ask for one token, which is enough to make Ollama load the model with a usual context window.
    request_body = {"model": model_name, "prompt": "Hi", "stream": False, "think": llm2.THINKING_ON,
                    "options": {"num_ctx": llm2.CONTEXT_WINDOW_BLOCK_IN_TOKENS, "num_predict": 1}}
    request = urllib.request.Request(settings.OLLAMA_ADDRESS + "/api/generate",
                                     data=json.dumps(request_body).encode("utf-8"),
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=900) as response:
        response.read()
    # Here we compare the model's memory on the GPU with its whole size; anything below 100% is on the CPU.
    with urllib.request.urlopen(settings.OLLAMA_ADDRESS + "/api/ps", timeout=10) as response:
        loaded_models = json.load(response)["models"]
    for model in loaded_models:
        if model["name"] == model_name:
            return round(100 * model["size_vram"] / model["size"])
    raise RuntimeError(f"{model_name} is not loaded after the memory check.")


# Here we start this project's server, unless it is already running, and remember whether we started it,
# so the last cell stops only a server this notebook started.
server_started_here = llm2.start_ollama_server()
for model_name in MODEL_NAMES:
    llm2.download_model(model_name)
check_the_models_are_the_macs()

# Here, on Kaggle, we stop now if the model doesn't fit on the GPU, before hours of slow runs.
if RUNNING_ON_KAGGLE:
    for model_name in MODEL_NAMES:
        share_on_gpu = gpu_share_of_model(model_name)
        print(f"Memory check: {model_name} is {share_on_gpu}% on the GPU")
        if share_on_gpu < 100:
            raise RuntimeError(f"{model_name} is only {share_on_gpu}% on the GPU. Stopping before slow runs.")

## 6. Questions and search index

The questions are built with the project's own sampling code and checked against the id lists on GitHub, so Kaggle
answers exactly the Mac's questions. The attached index is linked into `data/wiki/bm25_index/`, and a plain search of
every Pilot question must give exactly the Mac's results (`results/pilot/bm25_recall.csv` unchanged).

In [ ]:
from pipeline import dataset
from project_check import check_question_splits, check_search_index


def make_and_check_question_sets() -> None:
    """Build the Pilot and Test sets if they're missing, then stop if they differ from the committed id lists."""
    if not (dataset.questions_dir() / "pilot.jsonl").exists() or not (dataset.questions_dir() / "test.jsonl").exists():
        print("Building the Pilot and Test sets (downloads HotpotQA dev, 28 MB) ...")
        dataset.build()
    problems = check_question_splits(CONFIG)
    if problems:
        raise RuntimeError("The question sets don't match the committed id lists:\n" + "\n".join(problems))
    print("Same 100 Pilot and 1,000 Test questions as on the Mac.")


def link_the_attached_index() -> None:
    """On Kaggle, link the attached index dataset into data/wiki/bm25_index/ (no copying)."""
    from pipeline.retriever import index_dir
    index_link = index_dir()
    # Here we remove a broken link (its dataset was detached), so it can be linked again.
    if index_link.is_symlink() and not index_link.exists():
        index_link.unlink()
    if index_link.exists():
        print(f"Index already in place: {index_link}")
        return
    # Here we look for the settings file every saved bm25s index has, wherever Kaggle put the dataset.
    matches = sorted(Path("/kaggle/input").glob("**/params.index.json"))
    if not matches:
        raise RuntimeError("Search index not found. Add Input > Datasets > shoimyachy/hotpotqa-bm25-index, "
                           "then run again.")
    index_link.parent.mkdir(parents=True, exist_ok=True)
    index_link.symlink_to(matches[0].parent)
    print(f"Linked the attached index: {matches[0].parent}")


def check_search_matches_the_mac() -> None:
    """Search every Pilot question once, and stop if the results differ from the Mac's committed recall file."""
    from pipeline.retriever import Retriever, ranked_titles_for_questions, write_pilot_recall_csv
    pilot_questions = dataset.load_questions("pilot")
    ranked_titles = ranked_titles_for_questions(Retriever.load(), pilot_questions)
    write_pilot_recall_csv(pilot_questions, ranked_titles)
    # Here git compares the file just written with GitHub's copy; any difference means the search is not the Mac's.
    difference = subprocess.run(["git", "diff", "--exit-code", "--stat", "results/pilot/bm25_recall.csv"])
    if difference.returncode != 0:
        raise RuntimeError("Search results differ from the Mac's (results/pilot/bm25_recall.csv changed).")
    print("Search results match the Mac's.")


make_and_check_question_sets()
if RUNNING_ON_KAGGLE:
    link_the_attached_index()
index_problems = check_search_index()
if index_problems:
    raise RuntimeError("The search index is incomplete:\n" + "\n".join(index_problems))
check_search_matches_the_mac()

## 7. Run Condition D

This calls `run_condition_d()`, exactly as pressing Run on `pipeline/Methods/condition_d.py` does on the Mac. Each
question is written to the log as soon as it is answered, so a session that stops early keeps every finished question.

In [ ]:
from pipeline.Methods import condition_d

# Here we give Condition D this notebook's settings.
condition_d.QUESTION_SET = QUESTION_SET
condition_d.HOW_MANY_QUESTIONS = HOW_MANY_QUESTIONS
condition_d.LOG_FULL_PROMPTS = LOG_FULL_PROMPTS

# Here we run it; it returns its log file.
LOG_FILE = condition_d.run_condition_d()
print(f"\nLog: {LOG_FILE}")

## 8. Summary

In [ ]:
def print_summary_and_warnings(log_file: Path) -> None:
    """Print the log's summary block, then every WARNING line in the log, so problems can't hide in a long log."""
    log_lines = log_file.read_text(encoding="utf-8").splitlines()
    summary_start = log_lines.index("===== Summary =====")
    print("\n".join(log_lines[summary_start:]))

    warning_lines = [line for line in log_lines if "WARNING" in line]
    print(f"\nWarnings in the log: {len(warning_lines)}")
    for warning_line in warning_lines:
        print(f"  {warning_line.strip()}")


print_summary_and_warnings(LOG_FILE)

## 9. Stop the server and pack the log

In [ ]:
# Here we stop the server only if this notebook started it, never one someone else is using.
if server_started_here:
    llm2.stop_ollama_server()

# Here, on Kaggle, we zip this day's log folder into /kaggle/working, so the logs download as one file.
if RUNNING_ON_KAGGLE:
    date_folder = LOG_FILE.parent
    archive_path = shutil.make_archive(f"/kaggle/working/condition_d_logs_{date_folder.name}", "zip", date_folder)
    print(f"Logs packed into: {archive_path}")
else:
    print(f"Logs are in: {LOG_FILE.parent}")